<a href="https://colab.research.google.com/github/shahriariit/ML_Airlines_Services/blob/main/dataset_3_sentiment_analysis_and_model_classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install required libraries
#!pip install --upgrade pip
#!pip install tensorflow
#!pip install torch
#!pip install transformers
#!pip install tqdm
#!pip install nltk pandas

In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 9.5 MB/s eta 0:00:00


In [ ]:
import re
import numpy as np
import pandas as pd
import nltk
import torch
import tensorflow as tf

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import *
from sklearn.pipeline import Pipeline

from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    AdaBoostClassifier, VotingClassifier, StackingClassifier
)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.linear_model import SGDClassifier, PassiveAggressiveClassifier
from sklearn.calibration import CalibratedClassifierCV

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, GRU, Conv1D, GlobalMaxPooling1D, Dense, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import SpatialDropout1D, Dropout, BatchNormalization, MaxPooling1D, Input
from tensorflow.keras.callbacks import EarlyStopping
from transformers import (
    BertTokenizer, BertForSequenceClassification,
    Trainer, TrainingArguments
)
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.model_selection import RandomizedSearchCV
import optuna
from sklearn.model_selection import cross_val_score


# NLTK downloads
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('punkt_tab', quiet=True) # Added this line

True

In [ ]:
MAX_WORDS = 10000
MAX_LEN   = 100
EMBED_DIM = 100
TEST_SIZE = 0.2
VAL_SPLIT = 0.1
RANDOM_STATE = 42

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z\s]", "", text)

    tokens = nltk.word_tokenize(text)
    stop_words = set(nltk.corpus.stopwords.words('english'))
    lemmatizer = nltk.stem.WordNetLemmatizer()

    tokens = [lemmatizer.lemmatize(w) for w in tokens if w not in stop_words]
    return " ".join(tokens)

In [ ]:
def preprocess_dataframe(df):
    df = df.copy()
    df = df.drop(columns=['title', 'airline'], errors='ignore')

    print("Cleaning text...")
    df['cleaned_review'] = df['review'].astype(str).apply(clean_text)

    df = df.drop(columns=['review'], errors='ignore')

    le = LabelEncoder()
    df['target'] = le.fit_transform(df['target'])

    print("Label Encoding:", dict(zip(le.classes_, le.transform(le.classes_))))
    return df, le

In [ ]:
def split_data(df):
    X = df['cleaned_review']
    y = df['target']

    X_train, X_temp, y_train, y_temp = train_test_split(
        X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE
    )

    X_val, X_test, y_val, y_test = train_test_split(
        X_temp, y_temp, test_size=0.5, stratify=y_temp, random_state=RANDOM_STATE
    )

    return X_train, X_val, X_test, y_train, y_val, y_test

In [ ]:
def calculate_metrics(y_true, y_pred, y_proba=None, name="Model"):
    accuracy  = accuracy_score(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted')
    recall    = recall_score(y_true, y_pred, average='weighted')
    f1        = f1_score(y_true, y_pred, average='weighted')

    auc = np.nan

    if y_proba is not None and len(np.unique(y_true)) > 1:
        try:
            y_proba = np.array(y_proba)

            if y_proba.ndim == 1:
                y_proba = np.vstack([-y_proba, y_proba]).T
            if y_proba.shape[1] == 2:
                auc = roc_auc_score(y_true, y_proba[:, 1])
            else:
                auc = roc_auc_score(
                    y_true,
                    y_proba,
                    multi_class='ovr',
                    average='weighted'
                )

        except Exception:
            auc = np.nan

    print(f"\n{name}")
    print(f"Accuracy : {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall   : {recall:.4f}")
    print(f"F1 Score : {f1:.4f}")
    print(f"AUC      : {auc:.4f}")

    return {
        "Accuracy": accuracy,
        "Precision": precision,
        "Recall": recall,
        "F1 Score": f1,
        "AUC": auc
    }

In [ ]:
def run_classic_ml(X_train, X_test, y_train, y_test):

    vectorizer = TfidfVectorizer(max_features=MAX_WORDS, ngram_range=(1,2))

    X_train_vec = vectorizer.fit_transform(X_train)
    X_test_vec  = vectorizer.transform(X_test)

    models = {
        "LogReg": LogisticRegression(),
        "NaiveBayes": MultinomialNB(),
        "LinearSVM": LinearSVC(),
        "DecisionTree": DecisionTreeClassifier(max_depth=20, random_state=42),
        "RandomForest": RandomForestClassifier(n_estimators=200, random_state=42),
        "ExtraTrees": ExtraTreesClassifier( n_estimators=200, random_state=42),
        "GradientBoosting": GradientBoostingClassifier(n_estimators=200),
        "AdaBoost": AdaBoostClassifier(n_estimators=200),
        "KNN": KNeighborsClassifier(n_neighbors=5),
        "SGD": SGDClassifier(),
        "PassiveAggressive": PassiveAggressiveClassifier()

    }

    results = {}

    for idx, (name, model) in enumerate(models.items(), 1):
        print(f"\n[{idx}] Training {name}...")
        print("-"*60)

        model.fit(X_train_vec, y_train)

        y_pred = model.predict(X_test_vec)

        y_proba = None
        if hasattr(model, "predict_proba"):
            y_proba = model.predict_proba(X_test_vec)

        results[name] = calculate_metrics(y_test, y_pred, y_proba, name)

    return results

In [ ]:
def prepare_dl_data(X_train, X_val, X_test, y_train, y_val, y_test):

    tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
    tokenizer.fit_on_texts(X_train)

    X_train_seq = pad_sequences(tokenizer.texts_to_sequences(X_train), maxlen=MAX_LEN)
    X_val_seq   = pad_sequences(tokenizer.texts_to_sequences(X_val), maxlen=MAX_LEN)
    X_test_seq  = pad_sequences(tokenizer.texts_to_sequences(X_test), maxlen=MAX_LEN)

    return X_train_seq, X_val_seq, X_test_seq, y_train.values, y_val.values, y_test.values

In [ ]:
def create_lstm():
    return Sequential([
        Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
        LSTM(128),
        Dense(3, activation='softmax')
    ])

def create_gru():
    return Sequential([
        Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
        GRU(128),
        Dense(3, activation='softmax')
    ])

def create_cnn():
    return Sequential([
        Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
        Conv1D(128, 5, activation='relu'),
        GlobalMaxPooling1D(),
        Dense(3, activation='softmax')
    ])

def create_bilstm():
    return Sequential([
        Embedding(MAX_WORDS, EMBED_DIM, input_length=MAX_LEN),
        Bidirectional(LSTM(128)),
        Dense(3, activation='softmax')
    ])

In [ ]:
def train_dl_model(name, model_fn, X_train, y_train, X_test, y_test):

    print(f"\nTraining {name}...")

    model = model_fn()
    model.compile(loss='sparse_categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

    model.fit(X_train, y_train, epochs=5, batch_size=32, validation_split=VAL_SPLIT, verbose=0)

    y_proba = model.predict(X_test)
    y_pred  = np.argmax(y_proba, axis=1)

    metrics = calculate_metrics(y_test, y_pred, y_proba, name)
    return metrics

In [ ]:
print("Loading dataset...")
df = pd.read_csv("https://raw.githubusercontent.com/shahriariit/ML_Airlines_Services/refs/heads/main/merged_data.csv")

df, le = preprocess_dataframe(df)

X_train, X_val, X_test, y_train, y_val, y_test = split_data(df)

Loading dataset...
Cleaning text...
Label Encoding: {'Mixed': np.int64(0), 'Negative': np.int64(1), 'Positive': np.int64(2)}


In [ ]:
df.to_csv('cleaned_dataframe.csv', index=False)

In [ ]:
classic_ml_results = run_classic_ml(X_train, X_test, y_train, y_test)


[1] Training LogReg...
------------------------------------------------------------

LogReg
Accuracy : 0.8118
Precision: 0.8048
Recall   : 0.8118
F1 Score : 0.8069
AUC      : 0.9319

[2] Training NaiveBayes...
------------------------------------------------------------

NaiveBayes
Accuracy : 0.7688
Precision: 0.7569
Recall   : 0.7688
F1 Score : 0.7441
AUC      : 0.9145

[3] Training LinearSVM...
------------------------------------------------------------

LinearSVM
Accuracy : 0.7909
Precision: 0.7850
Recall   : 0.7909
F1 Score : 0.7874
AUC      : nan

[4] Training DecisionTree...
------------------------------------------------------------

DecisionTree
Accuracy : 0.6876
Precision: 0.6798
Recall   : 0.6876
F1 Score : 0.6826
AUC      : 0.7643

[5] Training RandomForest...
------------------------------------------------------------

RandomForest
Accuracy : 0.7626
Precision: 0.7565
Recall   : 0.7626
F1 Score : 0.7270
AUC      : 0.9160

[6] Training ExtraTrees...
----------------------

In [ ]:
X_train_dl, X_val_dl, X_test_dl, y_train_dl, y_val_dl, y_test_dl = prepare_dl_data(
    X_train, X_val, X_test, y_train, y_val, y_test
)

dl_models = {
    "LSTM": create_lstm,
    "GRU": create_gru,
    "CNN": create_cnn,
    "BiLSTM": create_bilstm
}
dl_results = {}
for name, fn in dl_models.items():
    metrics = train_dl_model(name, fn, X_train_dl, y_train_dl, X_test_dl, y_test_dl)
    dl_results[name] = metrics


Training LSTM...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 11ms/step

LSTM
Accuracy : 0.7232
Precision: 0.7347
Recall   : 0.7232
F1 Score : 0.7279
AUC      : 0.8754

Training GRU...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 7ms/step

GRU
Accuracy : 0.7134
Precision: 0.7048
Recall   : 0.7134
F1 Score : 0.7066
AUC      : 0.8720

Training CNN...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


26/26 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step

CNN
Accuracy : 0.7614
Precision: 0.7605
Recall   : 0.7614
F1 Score : 0.7609
AUC      : 0.9097

Training BiLSTM...


/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


26/26 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step

BiLSTM
Accuracy : 0.7355
Precision: 0.7310
Recall   : 0.7355
F1 Score : 0.7329
AUC      : 0.8743


In [ ]:
class TorchTextDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        # labels: pandas Series or np.array or list
        if hasattr(labels, 'values'):
            labels = labels.values
        self.labels = torch.tensor(labels, dtype=torch.long)
        self.encodings = encodings
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item
    def __len__(self):
        return len(self.labels)

In [ ]:
def pytorch_transformer_tokenize(tokenizer, texts, max_length):
    return tokenizer(texts.tolist(), truncation=True, padding=True, max_length=max_length)

def train_pytorch_transformer(
    model, tokenizer,
    X_train, y_train,
    X_val, y_val,
    X_test, y_test,
    label_encoder,
    max_length=128,
    model_name="Model"
):

    print(f"\n{'='*70}")
    print(f"TRAINING {model_name}")
    print(f"{'='*70}")

    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model.to(device)

    #  ensure correct classification setup
    model.config.problem_type = "single_label_classification"

    # Tokenization
    train_encodings = pytorch_transformer_tokenize(tokenizer, X_train, max_length)
    val_encodings   = pytorch_transformer_tokenize(tokenizer, X_val, max_length)
    test_encodings  = pytorch_transformer_tokenize(tokenizer, X_test, max_length)

    train_dataset = TorchTextDataset(train_encodings, y_train)
    val_dataset   = TorchTextDataset(val_encodings, y_val)
    test_dataset  = TorchTextDataset(test_encodings, y_test)

    # TrainingArguments
    training_args = TrainingArguments(
        output_dir=f"./results_{model_name.lower()}",
        num_train_epochs=3,

        per_device_train_batch_size=8,
        per_device_eval_batch_size=8,

        learning_rate=2e-5,
        warmup_ratio=0.1,

        eval_strategy="epoch",
        save_strategy="no",
        logging_strategy="epoch",

        max_grad_norm=1.0,

        report_to="none"
    )

    # Trainer
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=val_dataset
    )

    # Train
    trainer.train()

    # Predict
    test_outputs = trainer.predict(test_dataset)
    logits = test_outputs.predictions

    # Predictions
    y_pred = np.argmax(logits, axis=1)

    probs = torch.nn.functional.softmax(torch.tensor(logits), dim=1)
    probs = torch.nan_to_num(probs)
    y_proba = probs.cpu().numpy()

    # Evaluation
    results = calculate_metrics(
        y_test,
        y_pred,
        y_proba,
        model_name
    )

    return results

In [ ]:
transformer_models = {
    "BERT": "bert-base-uncased",
    "RoBERTa": "roberta-base",
    "DistilBERT": "distilbert-base-uncased",
    "ALBERT": "albert-base-v2",
    #"DeBERTa": "microsoft/deberta-v3-base"
}

In [ ]:
num_classes = len(le.classes_)
print(f"Number of classes: {num_classes}")

print("\n TRAINING TRANSFORMERS...")
print("="*70)
transformer_results = {}
for name, checkpoint in transformer_models.items():
    print(f"\n Loading {name}...")
    tokenizer = AutoTokenizer.from_pretrained(checkpoint)
    model = AutoModelForSequenceClassification.from_pretrained(
        checkpoint,
        num_labels=num_classes
    )
    metrics = train_pytorch_transformer(
        model,
        tokenizer,
        X_train, y_train,
        X_val, y_val,
        X_test, y_test,
        le,
        model_name=name
    )
    transformer_results[name] = metrics

Number of classes: 3

 TRAINING TRANSFORMERS...

 Loading BERT...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

...

AlbertForSequenceClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.bias             | UNEXPECTED | 
predictions.dense.bias       | UNEXPECTED | 
predictions.decoder.bias     | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
classifier.weight            | MISSING    | 
classifier.bias              | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



TRAINING ALBERT


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss
1,0.607699,0.488545
2,0.413396,0.402502
3,0.287162,0.524696



ALBERT
Accuracy : 0.8475
Precision: 0.8466
Recall   : 0.8475
F1 Score : 0.8467
AUC      : 0.9615


In [ ]:
all_results = {
    **classic_ml_results,
    **dl_results,
    **transformer_results
}

# Convert to DataFrame
results_df = pd.DataFrame.from_dict(all_results, orient='index')
results_df.index.name = 'Model'
print("\n--- Summary of Model Performance ---\n")
print(results_df.round(4))



--- Summary of Model Performance ---

                   Accuracy  Precision  Recall  F1 Score     AUC
Model                                                           
LogReg               0.8118     0.8048  0.8118    0.8069  0.9319
NaiveBayes           0.7688     0.7569  0.7688    0.7441  0.9145
LinearSVM            0.7909     0.7850  0.7909    0.7874     NaN
DecisionTree         0.6876     0.6798  0.6876    0.6826  0.7643
RandomForest         0.7626     0.7565  0.7626    0.7270  0.9160
ExtraTrees           0.7724     0.7862  0.7724    0.7336  0.9240
GradientBoosting     0.7601     0.7483  0.7601    0.7520  0.9130
AdaBoost             0.7269     0.7260  0.7269    0.7262  0.8770
KNN                  0.6814     0.6892  0.6814    0.6820  0.8394
SGD                  0.7847     0.7755  0.7847    0.7784     NaN
PassiveAggressive    0.7638     0.7638  0.7638    0.7638     NaN
LSTM                 0.7232     0.7347  0.7232    0.7279  0.8754
GRU                  0.7134     0.7048  0.7134    0

In [ ]:
top3_df = results_df.sort_values(by="AUC", ascending=False).head(3)

print("\n--- Top 3 Models ---\n")
print(top3_df)

top3_names = top3_df.index.tolist()


--- Top 3 Models ---

            Accuracy  Precision    Recall  F1 Score       AUC
Model                                                        
BERT        0.872079   0.871848  0.872079  0.871948  0.965113
DistilBERT  0.865929   0.864103  0.865929  0.864901  0.962489
RoBERTa     0.858549   0.857338  0.858549  0.857780  0.962062
